In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from itertools import product

In [ ]:
PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
OUT_DIR = PROJECT_DIR / "data" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
videos = sorted(RAW_DIR.glob("*.mp4"))
print("Found videos:", len(videos))
for v in videos[:10]:
    print(" -", v.name)

# picking the one with 1 particle
video_path = videos[7]
print("Using video:", video_path.name)

Found videos: 11
 - vid_001.mp4
 - vid_002.mp4
 - vid_003.mp4
 - vid_004.mp4
 - vid_005.mp4
 - vid_006.mp4
 - vid_007.mp4
 - vid_008.mp4
 - vid_009.mp4
 - vid_010.mp4
Using video: vid_008.mp4


In [ ]:
roi_df = pd.read_csv(OUT_DIR / "roi_table.csv")
row = roi_df.loc[roi_df["video_name"] == video_path.name].iloc[0]
roi = (int(row["roi_x"]), int(row["roi_y"]), int(row["roi_w"]), int(row["roi_h"]))
x0, y0, w, h = roi

print("Using ROI:", roi)

Using ROI: (550, 371, 363, 302)


In [ ]:
backSub = cv2.createBackgroundSubtractorMOG2(
    history=300,       # how many frames used to learn background
    varThreshold=16,   # sensitivity: lower = more foreground detected
    detectShadows=False
)

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

In [ ]:
# Helper: evaluate one parameter set on first N frames
# Returns summary metrics for auto-tuning

def eval_params_on_clip(
    video_path,
    roi,
    N=200,
    history=300,
    varThreshold=16,
    THRESH=180,
    MIN_AREA=30,
    MAX_AREA=5000,
    open_iter=1,
    close_iter=1,
    warmup=30,
):
    x0, y0, w, h = roi
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    backSub = cv2.createBackgroundSubtractorMOG2(
        history=history, varThreshold=varThreshold, detectShadows=False
    )
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

    counts = []
    frame_idx = 0

    while frame_idx < N:
        ret, frame = cap.read()
        if not ret:
            break

        roi_frame = frame[y0:y0+h, x0:x0+w]
        gray = cv2.cvtColor(roi_frame, cv2.COLOR_BGR2GRAY)

        fg = backSub.apply(gray)
        _, mask = cv2.threshold(fg, THRESH, 255, cv2.THRESH_BINARY)

        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=open_iter)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=close_iter)

        if frame_idx >= warmup:
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cnt_ok = 0
            for cnt in contours:
                area = cv2.contourArea(cnt)
                if MIN_AREA <= area <= MAX_AREA:
                    cnt_ok += 1
            counts.append(cnt_ok)

        frame_idx += 1

    cap.release()

    if len(counts) == 0:
        return None

    counts = np.array(counts, dtype=float)
    return {
        "varThreshold": varThreshold,
        "THRESH": THRESH,
        "MIN_AREA": MIN_AREA,
        "open_iter": open_iter,
        "close_iter": close_iter,
        "median_count": float(np.median(counts)),
        "mean_count": float(np.mean(counts)),
        "std_count": float(np.std(counts)),
        "zero_frac": float(np.mean(counts == 0)),
        "frames_used": int(len(counts)),
    }


In [ ]:
# Auto-tuning (grid search on short clip)
# This avoids manual guessing
# Still requires ONE quick visual check after

varThreshold_grid = [8, 16, 24]
THRESH_grid = [120, 160, 200]
MIN_AREA_grid = [20, 40, 80]
morph_grid = [(1, 1), (2, 1)]  # (open_iter, close_iter)

N_TUNE = 200
WARMUP = 30

results = []
for varT, thr, mina, (oi, ci) in product(varThreshold_grid, THRESH_grid, MIN_AREA_grid, morph_grid):
    r = eval_params_on_clip(
        video_path=video_path,
        roi=roi,
        N=N_TUNE,
        varThreshold=varT,
        THRESH=thr,
        MIN_AREA=mina,
        open_iter=oi,
        close_iter=ci,
        warmup=WARMUP,
    )
    if r is not None:
        results.append(r)

res_df = pd.DataFrame(results)

if len(res_df) == 0:
    raise RuntimeError("Auto-tuning produced no results. Check video/ROI, or reduce THRESH/MIN_AREA.")

# Ranking logic:
# - prefer fewer frames with zero detections (zero_frac low)
# - prefer stable detections (std_count low)
# - prefer not-crazy counts (median_count low-ish)
res_df = res_df.sort_values(["zero_frac", "std_count", "median_count"], ascending=[True, True, True])

print("Top 10 parameter candidates:")
display(res_df.head(10))

best = res_df.iloc[0].to_dict()
print("\nBest candidate picked (auto):", best)

Top 10 parameter candidates:


,varThreshold,THRESH,MIN_AREA,open_iter,close_iter,median_count,mean_count,std_count,zero_frac,frames_used
0,8,120,20,1,1,1.0,1.294118,0.619744,0.017647,170
6,8,160,20,1,1,1.0,1.294118,0.619744,0.017647,170
12,8,200,20,1,1,1.0,1.294118,0.619744,0.017647,170
2,8,120,40,1,1,1.0,1.147059,0.455835,0.035294,170
8,8,160,40,1,1,1.0,1.147059,0.455835,0.035294,170
14,8,200,40,1,1,1.0,1.147059,0.455835,0.035294,170
18,16,120,20,1,1,1.0,1.211765,0.585996,0.041176,170
24,16,160,20,1,1,1.0,1.211765,0.585996,0.041176,170
30,16,200,20,1,1,1.0,1.211765,0.585996,0.041176,170
36,24,120,20,1,1,1.0,1.188235,0.552941,0.047059,170



Best candidate picked (auto): {'varThreshold': 8.0, 'THRESH': 120.0, 'MIN_AREA': 20.0, 'open_iter': 1.0, 'close_iter': 1.0, 'median_count': 1.0, 'mean_count': 1.2941176470588236, 'std_count': 0.6197443384031023, 'zero_frac': 0.01764705882352941, 'frames_used': 170.0}


In [ ]:
# Visual preview of the mask + kept contours for one frame
# This is the "manual sanity check" step

def preview_mask_and_boxes(
    video_path,
    roi,
    varThreshold,
    THRESH,
    MIN_AREA,
    MAX_AREA=5000,
    open_iter=1,
    close_iter=1,
    frame_to_show=120
):
    x0, y0, w, h = roi
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError("Cannot open video for preview")

    backSub = cv2.createBackgroundSubtractorMOG2(history=300, varThreshold=varThreshold, detectShadows=False)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

    # Let the model "learn" up to frame_to_show
    gray = None
    fg = None
    for _ in range(frame_to_show + 1):
        ret, frame = cap.read()
        if not ret:
            break
        roi_frame = frame[y0:y0+h, x0:x0+w]
        gray = cv2.cvtColor(roi_frame, cv2.COLOR_BGR2GRAY)
        fg = backSub.apply(gray)

    cap.release()

    if gray is None or fg is None:
        raise RuntimeError("Preview failed: could not reach the requested frame.")

    _, mask = cv2.threshold(fg, THRESH, 255, cv2.THRESH_BINARY)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=open_iter)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=close_iter)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    kept = 0
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if MIN_AREA <= area <= MAX_AREA:
            kept += 1
            x, y, ww, hh = cv2.boundingRect(cnt)
            cv2.rectangle(vis, (x, y), (x + ww, y + hh), (0, 0, 255), 2)

    cv2.imshow("ROI gray", gray)
    cv2.imshow("Foreground mask (thresholded + cleaned)", mask)
    cv2.imshow(f"Kept contours={kept}", vis)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# Preview the best candidate:
preview_mask_and_boxes(
    video_path=video_path,
    roi=roi,
    varThreshold=int(best["varThreshold"]),
    THRESH=int(best["THRESH"]),
    MIN_AREA=int(best["MIN_AREA"]),
    open_iter=int(best["open_iter"]),
    close_iter=int(best["close_iter"]),
    frame_to_show=120
)

In [ ]:
# Run full motion detection with chosen params and save CSV
def run_motion_detection(
    video_path,
    roi,
    varThreshold,
    THRESH,
    MIN_AREA,
    MAX_AREA=5000,
    open_iter=1,
    close_iter=1,
    warmup=30,
    max_frames=None
):
    x0, y0, w, h = roi
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    backSub = cv2.createBackgroundSubtractorMOG2(history=300, varThreshold=varThreshold, detectShadows=False)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

    rows = []
    frame_idx = 0

    pbar_total = total if max_frames is None else min(total, max_frames)
    pbar = tqdm(total=pbar_total, desc="Running motion detection")

    while True:
        if max_frames is not None and frame_idx >= max_frames:
            break

        ret, frame = cap.read()
        if not ret:
            break

        roi_frame = frame[y0:y0+h, x0:x0+w]
        gray = cv2.cvtColor(roi_frame, cv2.COLOR_BGR2GRAY)

        fg = backSub.apply(gray)
        _, mask = cv2.threshold(fg, THRESH, 255, cv2.THRESH_BINARY)

        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=open_iter)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=close_iter)

        if frame_idx >= warmup:
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in contours:
                area = cv2.contourArea(cnt)
                if area < MIN_AREA or area > MAX_AREA:
                    continue

                M = cv2.moments(cnt)
                if M["m00"] == 0:
                    continue
                cx = M["m10"] / M["m00"]
                cy = M["m01"] / M["m00"]
                rows.append({"frame": frame_idx, "x": cx, "y": cy, "area": area})

        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()

    det_df = pd.DataFrame(rows)
    return det_df, fps, (w, h)

det_motion, fps, out_size = run_motion_detection(
    video_path=video_path,
    roi=roi,
    varThreshold=int(best["varThreshold"]),
    THRESH=int(best["THRESH"]),
    MIN_AREA=int(best["MIN_AREA"]),
    open_iter=int(best["open_iter"]),
    close_iter=int(best["close_iter"]),
    warmup=WARMUP,
    max_frames=None,
)

print("Detections rows:", len(det_motion))
display(det_motion.head())



Running motion detection: 100%|████████████████████████████████████████████████████| 2218/2218 [00:22<00:00, 97.37it/s]

Detections rows: 3664


,frame,x,y,area
0,30,230.703802,178.254642,188.5
1,30,218.181818,156.060606,22.0
2,31,226.716511,177.822430,53.5
3,31,233.763636,181.159596,82.5
4,31,218.284153,157.546448,30.5


In [ ]:
# Save CSV
csv_path = OUT_DIR / f"motion_detections_{video_path.stem}.csv"
det_motion.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)

# Quick stability check
if len(det_motion) > 0:
    counts = det_motion.groupby("frame").size()
    print("Median detections per frame:", float(counts.median()))
    print("Min/Max detections per frame:", int(counts.min()), int(counts.max()))
else:
    print("No detections in full run. Lower THRESH/MIN_AREA or lower varThreshold.")



Saved CSV: C:\Users\milli\Desktop\capstone_project\data\outputs\motion_detections_vid_008.csv
Median detections per frame: 1.0
Min/Max detections per frame: 1 6


In [ ]:
# Export annotated ROI video (circles at detected centroids)
annot_path = OUT_DIR / f"motion_annotated_{video_path.stem}.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(annot_path), fourcc, fps, out_size)

det_by_frame = {k: v for k, v in det_motion.groupby("frame")} if len(det_motion) else {}

cap = cv2.VideoCapture(str(video_path))
frame_idx = 0
pbar = tqdm(total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), desc="Writing annotated video")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    roi_frame = frame[y0:y0+h, x0:x0+w]

    if frame_idx in det_by_frame:
        for _, r in det_by_frame[frame_idx].iterrows():
            cv2.circle(roi_frame, (int(r["x"]), int(r["y"])), 6, (0, 0, 255), 2)

    writer.write(roi_frame)
    frame_idx += 1
    pbar.update(1)

pbar.close()
cap.release()
writer.release()

print("Saved annotated video:", annot_path)

Writing annotated video: 100%|████████████████████████████████████████████████████| 2218/2218 [00:18<00:00, 121.21it/s]


Saved annotated video: C:\Users\milli\Desktop\capstone_project\data\outputs\motion_annotated_vid_008.mp4
